# Dataset API

In [1]:
spark

Intitializing Scala interpreter ...

Spark Web UI available at http://28aa9a7c5449:4042
SparkContext available as 'sc' (version = 3.5.5, master = local[*], app id = local-1752276231253)
SparkSession available as 'spark'


res0: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@33c2c15b


In [ ]:
// If something is nullabe, you need to wrap the value type in Option[] - 
// this helps enforce assumptions about the pipeline

import org.apache.spark.sql.SparkSession 

val sparkSession = SparkSession.builder.appName("class-3-2").getOrCreate()

//TODO Illustrate how this fails if you change from Option[String] to String for referrer
case class Event (
   // Option is a way to handle NULL more gracefully
   // It replaces a command like ".where($"user_id".isNotNull)"
    user_id: Option[Integer], 
    device_id: Option[Integer],
    referrer: Option[String],
    host: String,
    url: String,
    event_time: String
)



In [10]:

val dummyData = List(
        Event(user_id=Some(1), device_id=Some(2), referrer=Some("linkedin"), host="eczachly.com", url="/signup", event_time="2023-01-01"),
        Event(user_id=Some(3), device_id=Some(7), referrer=Some("twitter"), host="eczachly.com", url="/signup", event_time="2023-01-01")
    )




dummyData: List[Event] = List(Event(Some(1),Some(2),Some(linkedin),eczachly.com,/signup,2023-01-01), Event(Some(3),Some(7),Some(twitter),eczachly.com,/signup,2023-01-01))


In [13]:
// best practices - define the output

val dummyData: List[Event] = List(
        Event(user_id=Some(1), device_id=Some(2), referrer=Some("linkedin"), host="eczachly.com", url="/signup", event_time="2023-01-01"),
        Event(user_id=Some(3), device_id=Some(7), referrer=Some("twitter"), host="eczachly.com", url="/signup", event_time="2023-01-01")
    )

dummyData: List[Event] = List(Event(Some(1),Some(2),Some(linkedin),eczachly.com,/signup,2023-01-01), Event(Some(3),Some(7),Some(twitter),eczachly.com,/signup,2023-01-01))


In [14]:

//TODO Illustrate how this fails if you change from Option[Long] to Long
case class Device (
    device_id: Integer,
    browser_type: String,
    os_type: String,
    device_type: String
)

case class EventWithDeviceInfo (
   user_id: Integer,
    device_id: Integer,
    browser_type: String,
    os_type: String,
    device_type: String,
    referrer: String,
    host: String,
    url: String,
    event_time: String
)

defined class Device
defined class EventWithDeviceInfo


In [15]:

import org.apache.spark.sql.Dataset

import sparkSession.implicits._

// Applying this case class before hand is very powerful, enforces Nullability/non-nullability at runtime!
val events: Dataset[Event] = sparkSession.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/events.csv")
                        .as[Event]

val devices: Dataset[Device] = sparkSession.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/devices.csv")
                        .as[Device]

devices.createOrReplaceTempView("devices")
events.createOrReplaceTempView("events")


import org.apache.spark.sql.Dataset
import sparkSession.implicits._
events: org.apache.spark.sql.Dataset[Event] = [user_id: int, device_id: int ... 4 more fields]
devices: org.apache.spark.sql.Dataset[Device] = [device_id: int, browser_type: string ... 2 more fields]


In [ ]:

// For simple transformations, you can see that these approaches are very similar. 
// Dataset is winning slightly because of the quality enforcement
val filteredViaDataset = events.filter(event => event.user_id.isDefined && event.device_id.isDefined)
val filteredViaDataFrame = events.toDF().where($"user_id".isNotNull && $"device_id".isNotNull)
val filteredViaSparkSql = sparkSession.sql("SELECT * FROM events WHERE user_id IS NOT NULL AND device_id IS NOT NULL")


In [20]:
// For simple transformations, you can see that these approaches are very similar. 
// Dataset is winning slightly because of the quality enforcement

val filteredViaDataset = events.filter(event => event.user_id.isDefined && event.device_id.isDefined)

filteredViaDataset: org.apache.spark.sql.Dataset[Event] = [user_id: int, device_id: int ... 4 more fields]


In [21]:
// For simple transformations, you can see that these approaches are very similar. 
// Dataset is winning slightly because of the quality enforcement

val filteredViaDataFrame = events.toDF().where($"user_id".isNotNull && $"device_id".isNotNull)

filteredViaDataFrame: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, device_id: int ... 4 more fields]


In [22]:
// For simple transformations, you can see that these approaches are very similar. 
// Dataset is winning slightly because of the quality enforcement

val filteredViaSparkSql = sparkSession.sql("SELECT * FROM events WHERE user_id IS NOT NULL AND device_id IS NOT NULL")

filteredViaSparkSql: org.apache.spark.sql.DataFrame = [user_id: int, device_id: int ... 4 more fields]


In [24]:
// Started with the simplest join - using sql

//Creating temp views is a good strategy if you're leveraging SparkSQL

filteredViaSparkSql.createOrReplaceTempView("filtered_events")
val combinedViaSparkSQL = spark.sql(f"""
    SELECT 
        fe.user_id,
        d.device_id,
        d.browser_type,
        d.os_type,
        d.device_type,
        fe. referrer,
        fe.host,
        fe.url,
        fe.event_time
    FROM filtered_events fe 
    JOIN devices d ON fe.device_id = d.device_id
""")

combinedViaSparkSQL.show(5)

+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------------+
|    user_id|device_id|browser_type|os_type|device_type|referrer|                host|url|          event_time|
+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------------+
| 1037710827|532630305|       Other|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-03-08 17:27:...|
|  925588856|532630305|       Other|  Other|      Other|    NULL|    www.eczachly.com|  /|2021-05-10 11:26:...|
|-1180485268|532630305|       Other|  Other|      Other|    NULL|admin.zachwilson....|  /|2021-02-17 16:19:...|
|-1044833855|532630305|       Other|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-09-24 15:53:...|
|  747494706|532630305|       Other|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-09-26 16:03:...|
+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------

combinedViaSparkSQL: org.apache.spark.sql.DataFrame = [user_id: int, device_id: int ... 7 more fields]


In [26]:
// DataFrames give up some of the intellisense because you no longer have static typing
val combinedViaDataFrames = filteredViaDataFrame.as("e")
//Make sure to use triple equals when using data frames
.join(devices.as("d"), $"e.device_id" === $"d.device_id", "inner")
.select(
  $"e.user_id",
  $"d.device_id",
  col("d.browser_type"), // both syntax $ or col works
  $"d.os_type",
  $"d.device_type",
  $"e.referrer",
  $"e.host",
  $"e.url",
  $"e.event_time"
)

combinedViaDataFrames.show(5)

+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------------+
|    user_id|device_id|browser_type|os_type|device_type|referrer|                host|url|          event_time|
+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------------+
| 1037710827|532630305|       Other|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-03-08 17:27:...|
|  925588856|532630305|       Other|  Other|      Other|    NULL|    www.eczachly.com|  /|2021-05-10 11:26:...|
|-1180485268|532630305|       Other|  Other|      Other|    NULL|admin.zachwilson....|  /|2021-02-17 16:19:...|
|-1044833855|532630305|       Other|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-09-24 15:53:...|
|  747494706|532630305|       Other|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-09-26 16:03:...|
+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------

combinedViaDataFrames: org.apache.spark.sql.DataFrame = [user_id: int, device_id: int ... 7 more fields]


In [ ]:

// This will fail if user_id is None

val combinedViaDatasets = filteredViaDataset
    .joinWith(devices, events("device_id") === devices("device_id"), "inner")
    .map{ case (event: Event, device: Device) => EventWithDeviceInfo(
                  user_id=event.user_id.get,
                  device_id=device.device_id,
                  browser_type=device.browser_type,
                  os_type=device.os_type,
                  device_type=device.device_type,
                  referrer=event.referrer.getOrElse("unknow"),
                  host=event.host,
                  url=event.url,
                  event_time=event.event_time
              ) }
  .map { eventWithDevice =>
        // Convert browser_type to uppercase while maintaining immutability
        eventWithDevice.copy(browser_type = eventWithDevice.browser_type.toUpperCase)
    }


In [28]:
val combinedViaDatasets = filteredViaDataset
.joinWith(devices, events("device_id") === devices("device_id"), "inner")
.map { case (event: Event, device: Device) => EventWithDeviceInfo (
    user_id = event.user_id.get,
    device_id = device.device_id,
    browser_type = device.browser_type,
    os_type = device.os_type,
    device_type = device.device_type,
    referrer = event.referrer.getOrElse("unknown"),
    host = event.host,
    url = event.url,
    event_time = event.event_time
    ) 
}

<console>:45: warning: fruitless type test: a value of type Event cannot also be a Event
       .map { case (event: Event, device: Device) => EventWithDeviceInfo (
                           ^
<console>:45: warning: match may not be exhaustive.
It would fail on the following input: (Event(_, _, _, _, _, _), Device(_, _, _, _))
       .map { case (event: Event, device: Device) => EventWithDeviceInfo (
            ^
combinedViaDatasets: org.apache.spark.sql.Dataset[EventWithDeviceInfo] = [user_id: int, device_id: int ... 7 more fields]


In [29]:
val combinedViaDatasets = filteredViaDataset
.joinWith(devices, events("device_id") === devices("device_id"), "inner")
.map { case (event, device: Device) => EventWithDeviceInfo (
    user_id = event.user_id.get,
    device_id = device.device_id,
    browser_type = device.browser_type,
    os_type = device.os_type,
    device_type = device.device_type,
    referrer = event.referrer.getOrElse("unknown"),
    host = event.host,
    url = event.url,
    event_time = event.event_time
    ) 
}

combinedViaDatasets: org.apache.spark.sql.Dataset[EventWithDeviceInfo] = [user_id: int, device_id: int ... 7 more fields]


In [30]:
val combinedViaDatasets = filteredViaDataset
.joinWith(devices, events("device_id") === devices("device_id"), "inner")
.map { case (event: Event, device: Device) => EventWithDeviceInfo (
    user_id = event.user_id.get,
    device_id = device.device_id,
    browser_type = device.browser_type,
    os_type = device.os_type,
    device_type = device.device_type,
    referrer = event.referrer.getOrElse("unknown"),
    host = event.host,
    url = event.url,
    event_time = event.event_time
    ) 
}

<console>:45: warning: fruitless type test: a value of type Event cannot also be a Event
       .map { case (event: Event, device: Device) => EventWithDeviceInfo (
                           ^
<console>:45: warning: match may not be exhaustive.
It would fail on the following input: (Event(_, _, _, _, _, _), Device(_, _, _, _))
       .map { case (event: Event, device: Device) => EventWithDeviceInfo (
            ^
combinedViaDatasets: org.apache.spark.sql.Dataset[EventWithDeviceInfo] = [user_id: int, device_id: int ... 7 more fields]


In [40]:
val combinedViaDatasets = filteredViaDataset
.joinWith(devices, events("device_id") === devices("device_id"), "inner")
.map { case (event, device: Device) => EventWithDeviceInfo (
    user_id = event.user_id.get,
    device_id = device.device_id,
    browser_type = device.browser_type,
    os_type = device.os_type,
    device_type = device.device_type,
    referrer = event.referrer.getOrElse("unknown"),
    host = event.host,
    url = event.url,
    event_time = event.event_time
    ) 
}

val rows = combinedViaDatasets.take(5)
rows.foreach(println)

EventWithDeviceInfo(1037710827,532630305,Other,Other,Other,unknown,www.zachwilson.tech,/,2021-03-08 17:27:24.241)
EventWithDeviceInfo(925588856,532630305,Other,Other,Other,unknown,www.eczachly.com,/,2021-05-10 11:26:21.247)
EventWithDeviceInfo(-1180485268,532630305,Other,Other,Other,unknown,admin.zachwilson.tech,/,2021-02-17 16:19:30.738)
EventWithDeviceInfo(-1044833855,532630305,Other,Other,Other,unknown,www.zachwilson.tech,/,2021-09-24 15:53:14.466)
EventWithDeviceInfo(747494706,532630305,Other,Other,Other,unknown,www.zachwilson.tech,/,2021-09-26 16:03:17.535)


combinedViaDatasets: org.apache.spark.sql.Dataset[EventWithDeviceInfo] = [user_id: int, device_id: int ... 7 more fields]
rows: Array[EventWithDeviceInfo] = Array(EventWithDeviceInfo(1037710827,532630305,Other,Other,Other,unknown,www.zachwilson.tech,/,2021-03-08 17:27:24.241), EventWithDeviceInfo(925588856,532630305,Other,Other,Other,unknown,www.eczachly.com,/,2021-05-10 11:26:21.247), EventWithDeviceInfo(-1180485268,532630305,Other,Other,Other,unknown,admin.zachwilson.tech,/,2021-02-17 16:19:30.738), EventWithDeviceInfo(-1044833855,532630305,Other,Other,Other,unknown,www.zachwilson.tech,/,2021-09-24 15:53:14.466), EventWithDeviceInfo(747494706,532630305,Other,Other,Other,unknown,www.zachwilson.tech,/,2021-09-26 16:03:17.535))


In [34]:
val combinedViaDatasets = filteredViaDataset
.joinWith(devices, events("device_id") === devices("device_id"), "inner")
.map { case (event: Event, device: Device) => EventWithDeviceInfo (
    user_id = event.user_id.get,
    device_id = device.device_id,
    browser_type = device.browser_type,
    os_type = device.os_type,
    device_type = device.device_type,
    referrer = event.referrer.getOrElse("unknown"),
    host = event.host,
    url = event.url,
    event_time = event.event_time
    ) 
}

val rows = combinedViaDatasets.take(5)
rows.foreach(println)

<console>: 47: warning: fruitless type test: a value of type Event cannot also be a Event

In [31]:
val combinedViaDatasets2 = combinedViaDatasets
.map {
 eventWithDevice =>
 // convert browser_type to uppercase while maintaining immutability
 eventWithDevice.copy(browser_type = eventWithDevice.browser_type.toUpperCase)
}

combinedViaDatasets2: org.apache.spark.sql.Dataset[EventWithDeviceInfo] = [user_id: int, device_id: int ... 7 more fields]


In [45]:
val combinedViaDatasets2 = combinedViaDatasets
.map { case (row: eventWithDeviceInfo) => {
        row.browser_type = toUpperCase(row.browser_type)
        return row
    }}
// taught in class

<console>: 36: error: not found: type eventWithDeviceInfo

In [47]:
def toUpperCase2(s: String): String = {
    return s.toUpperCase()
}

val toUpperCaseUdf = udf(toUpperCase2 _) // the _ indicates you passing a function as a value

toUpperCase2: (s: String)String
toUpperCaseUdf: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction($Lambda$5239/0x00007525dd441000@69c7e836,StringType,List(Some(class[value[0]: string])),Some(class[value[0]: string]),None,true,true)


In [49]:
// DataFrames give up some of the intellisense because you no longer have static typing
val combinedViaDataFrames = filteredViaDataFrame.as("e")
//Make sure to use triple equals when using data frames
.join(devices.as("d"), $"e.device_id" === $"d.device_id", "inner")
.select(
  $"e.user_id",
  $"d.device_id",
  toUpperCaseUdf($"d.browser_type").as("browser_type"), 
  $"d.os_type",
  $"d.device_type",
  $"e.referrer",
  $"e.host",
  $"e.url",
  $"e.event_time"
)

combinedViaDataFrames.show(5)

+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------------+
|    user_id|device_id|browser_type|os_type|device_type|referrer|                host|url|          event_time|
+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------------+
| 1037710827|532630305|       OTHER|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-03-08 17:27:...|
|  925588856|532630305|       OTHER|  Other|      Other|    NULL|    www.eczachly.com|  /|2021-05-10 11:26:...|
|-1180485268|532630305|       OTHER|  Other|      Other|    NULL|admin.zachwilson....|  /|2021-02-17 16:19:...|
|-1044833855|532630305|       OTHER|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-09-24 15:53:...|
|  747494706|532630305|       OTHER|  Other|      Other|    NULL| www.zachwilson.tech|  /|2021-09-26 16:03:...|
+-----------+---------+------------+-------+-----------+--------+--------------------+---+--------------

combinedViaDataFrames: org.apache.spark.sql.DataFrame = [user_id: int, device_id: int ... 7 more fields]


# Caching

In [1]:
import org.apache.spark.sql.functions._
import org.apache.spark.storage.StorageLevel

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.analyzer.failAmbiguousSelfJoin", "false")
spark.conf.set("spark.sql.shuffle.partitions", "4")

Intitializing Scala interpreter ...

Spark Web UI available at http://28aa9a7c5449:4042
SparkContext available as 'sc' (version = 3.5.5, master = local[*], app id = local-1752287313791)
SparkSession available as 'spark'


import org.apache.spark.sql.functions._
import org.apache.spark.storage.StorageLevel


In [2]:
val users = spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/events.csv")
                        .where($"user_id".isNotNull)

users.createOrReplaceTempView("events")

users: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, device_id: int ... 4 more fields]


In [3]:
val devices = spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/devices.csv")

devices.createOrReplaceTempView("devices")

devices: org.apache.spark.sql.DataFrame = [device_id: int, browser_type: string ... 2 more fields]


In [4]:
val executionDate = "2023-01-01"

// Caching here should be < 5 GBs or used for broadcast join
// You need to tune executor memory otherwise itll spill to disk and be slow
// Dont really try using any of the other StorageLevel besides MEMORY_ONLY

val eventsAggregated = spark.sql(f"""
  SELECT user_id, 
          device_id, 
        COUNT(1) as event_counts, 
        COLLECT_LIST(DISTINCT host) as host_array
  FROM events
  GROUP BY 1,2
""")


executionDate: String = 2023-01-01
eventsAggregated: org.apache.spark.sql.DataFrame = [user_id: int, device_id: int ... 2 more fields]


In [ ]:
// eventsAggregated.write.mode("overwrite").saveAsTable("bootcamp.events_aggregated_staging")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bootcamp.events_aggregated_staging (
        user_id BIGINT,
        device_id BIGINT,
        event_counts BIGINT,
        host_array ARRAY<STRING>
    )
    PARTITIONED BY (ds STRING)
""")



In [5]:
val usersAndDevices = users
  .join(eventsAggregated, eventsAggregated("user_id") === users("user_id"))
  .groupBy(users("user_id"))
  .agg(
    users("user_id"),
    max(eventsAggregated("event_counts")).as("total_hits"),
    collect_list(eventsAggregated("device_id")).as("devices")
  )

val devicesOnEvents = devices
      .join(eventsAggregated, devices("device_id") === eventsAggregated("device_id"))
      .groupBy(devices("device_id"), devices("device_type"))
      .agg(
        devices("device_id"),
        devices("device_type"),
         collect_list(eventsAggregated("user_id")).as("users")
      )

devicesOnEvents.explain()
usersAndDevices.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ObjectHashAggregate(keys=[device_id#47, device_type#50], functions=[collect_list(user_id#17, 0, 0)])
   +- ObjectHashAggregate(keys=[device_id#47, device_type#50], functions=[partial_collect_list(user_id#17, 0, 0)])
      +- Project [device_id#47, device_type#50, user_id#17]
         +- SortMergeJoin [device_id#47], [device_id#18], Inner
            :- Sort [device_id#47 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(device_id#47, 4), ENSURE_REQUIREMENTS, [plan_id=92]
            :     +- Filter isnotnull(device_id#47)
            :        +- FileScan csv [device_id#47,device_type#50] Batched: false, DataFilters: [isnotnull(device_id#47)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/devices.csv], PartitionFilters: [], PushedFilters: [IsNotNull(device_id)], ReadSchema: struct<device_id:int,device_type:string>
            +- Sort [device_id#18 ASC NULLS FIRST], false, 0
      

usersAndDevices: org.apache.spark.sql.DataFrame = [user_id: int, user_id: int ... 2 more fields]
devicesOnEvents: org.apache.spark.sql.DataFrame = [device_id: int, device_type: string ... 3 more fields]


In [6]:
devicesOnEvents.take(1)
usersAndDevices.take(1)

res4: Array[org.apache.spark.sql.Row] = Array([-2147470439,-2147470439,3,WrappedArray(378988111, 378988111, 378988111)])


In [7]:
val executionDate = "2023-01-01"

//Caching here should be < 5 GBs or used for broadcast join
//You need to tune executor memory otherwise itll spill to disk and be slow
//Dont really try using any of the other StorageLevel besides MEMORY_ONLY

val eventsAggregated = spark.sql(f"""
  SELECT user_id, 
          device_id, 
        COUNT(1) as event_counts, 
        COLLECT_LIST(DISTINCT host) as host_array
  FROM events
  GROUP BY 1,2
""").cache()

val usersAndDevices = users
  .join(eventsAggregated, eventsAggregated("user_id") === users("user_id"))
  .groupBy(users("user_id"))
  .agg(
    users("user_id"),
    max(eventsAggregated("event_counts")).as("total_hits"),
    collect_list(eventsAggregated("device_id")).as("devices")
  )

val devicesOnEvents = devices
      .join(eventsAggregated, devices("device_id") === eventsAggregated("device_id"))
      .groupBy(devices("device_id"), devices("device_type"))
      .agg(
        devices("device_id"),
        devices("device_type"),
         collect_list(eventsAggregated("user_id")).as("users")
      )

devicesOnEvents.explain()
usersAndDevices.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ObjectHashAggregate(keys=[device_id#47, device_type#50], functions=[collect_list(user_id#17, 0, 0)])
   +- ObjectHashAggregate(keys=[device_id#47, device_type#50], functions=[partial_collect_list(user_id#17, 0, 0)])
      +- Project [device_id#47, device_type#50, user_id#17]
         +- SortMergeJoin [device_id#47], [device_id#18], Inner
            :- Sort [device_id#47 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(device_id#47, 4), ENSURE_REQUIREMENTS, [plan_id=687]
            :     +- Filter isnotnull(device_id#47)
            :        +- FileScan csv [device_id#47,device_type#50] Batched: false, DataFilters: [isnotnull(device_id#47)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/devices.csv], PartitionFilters: [], PushedFilters: [IsNotNull(device_id)], ReadSchema: struct<device_id:int,device_type:string>
            +- Sort [device_id#18 ASC NULLS FIRST], false, 0
     

executionDate: String = 2023-01-01
eventsAggregated: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, device_id: int ... 2 more fields]
usersAndDevices: org.apache.spark.sql.DataFrame = [user_id: int, user_id: int ... 2 more fields]
devicesOnEvents: org.apache.spark.sql.DataFrame = [device_id: int, device_type: string ... 3 more fields]


## Types of caching

In [8]:
val eventsAggregated = spark.sql(f"""
  SELECT user_id, 
          device_id, 
        COUNT(1) as event_counts, 
        COLLECT_LIST(DISTINCT host) as host_array
  FROM events
  GROUP BY 1,2
""").cache()

eventsAggregated.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [user_id#17, device_id#18, event_counts#417L, host_array#418]
      +- InMemoryRelation [user_id#17, device_id#18, event_counts#417L, host_array#418], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=false
               +- ObjectHashAggregate(keys=[user_id#17, device_id#18], functions=[count(1), collect_list(distinct host#20, 0, 0)])
                  +- Exchange hashpartitioning(user_id#17, device_id#18, 4), ENSURE_REQUIREMENTS, [plan_id=651]
                     +- ObjectHashAggregate(keys=[user_id#17, device_id#18], functions=[merge_count(1), partial_collect_list(distinct host#20, 0, 0)])
                        +- HashAggregate(keys=[user_id#17, device_id#18, host#20], functions=[merge_count(1)])
                           +- Exchange hashpartitioning(user_id#17, device_id#18, host#20, 4), ENSURE_REQUIREMENTS, [plan_id=647]
                              +- 

eventsAggregated: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, device_id: int ... 2 more fields]


In [9]:
val eventsAggregated = spark.sql(f"""
  SELECT user_id, 
          device_id, 
        COUNT(1) as event_counts, 
        COLLECT_LIST(DISTINCT host) as host_array
  FROM events
  GROUP BY 1,2
""").cache(StorageLevel.MEMORY_ONLY)

eventsAggregated.explain()

<console>: 35: error: no arguments allowed for nullary method cache: ()org.apache.spark.sql.Dataset[org.apache.spark.sql.Row]

In [10]:
val eventsAggregated = spark.sql(f"""
  SELECT user_id, 
          device_id, 
        COUNT(1) as event_counts, 
        COLLECT_LIST(DISTINCT host) as host_array
  FROM events
  GROUP BY 1,2
""").persist(StorageLevel.MEMORY_ONLY)

eventsAggregated.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [user_id#17, device_id#18, event_counts#485L, host_array#486]
      +- InMemoryRelation [user_id#17, device_id#18, event_counts#485L, host_array#486], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=false
               +- ObjectHashAggregate(keys=[user_id#17, device_id#18], functions=[count(1), collect_list(distinct host#20, 0, 0)])
                  +- Exchange hashpartitioning(user_id#17, device_id#18, 4), ENSURE_REQUIREMENTS, [plan_id=651]
                     +- ObjectHashAggregate(keys=[user_id#17, device_id#18], functions=[merge_count(1), partial_collect_list(distinct host#20, 0, 0)])
                        +- HashAggregate(keys=[user_id#17, device_id#18, host#20], functions=[merge_count(1)])
                           +- Exchange hashpartitioning(user_id#17, device_id#18, host#20, 4), ENSURE_REQUIREMENTS, [plan_id=647]
                              +- 

eventsAggregated: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, device_id: int ... 2 more fields]


In [11]:
val eventsAggregated = spark.sql(f"""
  SELECT user_id, 
          device_id, 
        COUNT(1) as event_counts, 
        COLLECT_LIST(DISTINCT host) as host_array
  FROM events
  GROUP BY 1,2
""").persist(StorageLevel.DISK_ONLY)

// if you re going to use persist(StorageLevel.DISK_ONLY) to cache in disk, it s better to use .write.mode("overwrite")
// in caso the kernel restarts, one can still access the dataframe getting from table

eventsAggregated.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [user_id#17, device_id#18, event_counts#553L, host_array#554]
      +- InMemoryRelation [user_id#17, device_id#18, event_counts#553L, host_array#554], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=false
               +- ObjectHashAggregate(keys=[user_id#17, device_id#18], functions=[count(1), collect_list(distinct host#20, 0, 0)])
                  +- Exchange hashpartitioning(user_id#17, device_id#18, 4), ENSURE_REQUIREMENTS, [plan_id=651]
                     +- ObjectHashAggregate(keys=[user_id#17, device_id#18], functions=[merge_count(1), partial_collect_list(distinct host#20, 0, 0)])
                        +- HashAggregate(keys=[user_id#17, device_id#18, host#20], functions=[merge_count(1)])
                           +- Exchange hashpartitioning(user_id#17, device_id#18, host#20, 4), ENSURE_REQUIREMENTS, [plan_id=647]
                              +- 

eventsAggregated: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [user_id: int, device_id: int ... 2 more fields]


In [12]:
eventsAggregated.unpersist()

res10: eventsAggregated.type = [user_id: int, device_id: int ... 2 more fields]


# Bucket joins

In [1]:
import org.apache.spark.sql.SparkSession 

val sparkSession = SparkSession.builder.appName("class-3-2-bucket-join").getOrCreate()

Intitializing Scala interpreter ...

Spark Web UI available at http://28aa9a7c5449:4040
SparkContext available as 'sc' (version = 3.5.5, master = local[*], app id = local-1752322932074)
SparkSession available as 'spark'


import org.apache.spark.sql.SparkSession
sparkSession: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@69b39e4a


In [2]:
// In python use: from pyspark.sql.functions import broadcast, split, lit
import org.apache.spark.sql.functions.{broadcast, split, lit}


val matchesBucketed = spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/matches.csv")


val matchDetailsBucketed =  spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/match_details.csv")

import org.apache.spark.sql.functions.{broadcast, split, lit}
matchesBucketed: org.apache.spark.sql.DataFrame = [match_id: string, mapid: string ... 8 more fields]
matchDetailsBucketed: org.apache.spark.sql.DataFrame = [match_id: string, player_gamertag: string ... 34 more fields]


In [4]:
println(matchesBucketed.count(), matchDetailsBucketed.count())

(24025,151761)


In [5]:
spark.sql("""DROP TABLE IF EXISTS bootcamp.matches_bucketed""")
val bucketedDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
     match_id STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
 USING iceberg
 PARTITIONED BY (completion_date, bucket(16, match_id));
 """
 spark.sql(bucketedDDL)

bucketedDDL: String =
"
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
     match_id STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
 USING iceberg
 PARTITIONED BY (completion_date, bucket(16, match_id));
 "
res2: org.apache.spark.sql.DataFrame = []


In [7]:
val q = """SHOW TABLES IN bootcamp"""
spark.sql(q).show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
| bootcamp|              events|      false|
| bootcamp|       events_sorted|      false|
| bootcamp|     events_unsorted|      false|
| bootcamp|match_details_buc...|      false|
| bootcamp|    matches_bucketed|      false|
+---------+--------------------+-----------+



bucketedDDL: String =
"
SHOW TABLES IN bootcamp
 "


In [3]:
// out of memory, even with limit(4096)

matchesBucketed.select(
    $"match_id", $"is_team_game", $"playlist_id", $"completion_date"
    )
    .write.mode("append")
    .partitionBy("completion_date")
  // .bucketBy(16, "match_id")
  .saveAsTable("bootcamp.matches_bucketed")

java.lang.IllegalArgumentException:  requirement failed: The provided partitioning does not match of the table.

Deepseek explanation about OOO:

Spark and Iceberg: Overcoming OOM with Bucketing
When working with Spark and Iceberg to create tables with bucketing, you may encounter Out Of Memory (OOM) errors when using too many buckets (like 16 in your case). Here's what's happening and how to address it:

Understanding the Issue
Bucketing Overhead: Each bucket requires its own file writer in memory during the write operation

Memory Pressure: With 16 buckets, Spark needs to maintain 16 writers simultaneously, which can exhaust memory

Partition Handling: Iceberg tracks each file/bucket separately, adding metadata overhead

In [3]:
spark.sql("""DROP TABLE IF EXISTS bootcamp.matches_bucketed""")
val bucketedDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
     match_id STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
 USING iceberg
 PARTITIONED BY (completion_date, bucket(2, match_id));
 """
 spark.sql(bucketedDDL)

bucketedDDL: String =
"
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
     match_id STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
 USING iceberg
 PARTITIONED BY (completion_date, bucket(2, match_id));
 "
res0: org.apache.spark.sql.DataFrame = []


In [ ]:
// with 4 buckets also failed
// neither 2 buckets - memory constraints (the log shows up in the terminal - my case, IntelliJ)

matchesBucketed.select(
    $"match_id", $"is_team_game", $"playlist_id", $"completion_date"
    )
    .write.mode("append")
    .partitionBy("completion_date")
   .bucketBy(2, "match_id")
  .saveAsTable("bootcamp.matches_bucketed")

In [6]:
val bucketedDetailsDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketedDetailsDDL)

bucketedDetailsDDL: String =
"
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    player_total_kills INTEGER,
    player_total_deaths INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"
res3: org.apache.spark.sql.DataFrame = []


In [ ]:
matchDetailsBucketed.select(
    $"match_id", $"player_gamertag", $"player_total_kills", $"player_total_deaths")
    .write.mode("append")
  .bucketBy(16, "match_id").saveAsTable("bootcamp.match_details_bucketed")

In [11]:
val analysisQuery = """
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'matches_bucketed' as table
FROM demo.bootcamp.matches_bucketed.files

UNION ALL

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'match_details_bucketed' as table
FROM demo.bootcamp.match_details_bucketed.files
"""

spark.sql(analysisQuery).show()

+----+---------+--------------------+
|size|num_files|               table|
+----+---------+--------------------+
|NULL|        0|    matches_bucketed|
|NULL|        0|match_details_buc...|
+----+---------+--------------------+



analysisQuery: String =
"
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'matches_bucketed' as table
FROM demo.bootcamp.matches_bucketed.files

UNION ALL

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'match_details_bucketed' as table
FROM demo.bootcamp.match_details_bucketed.files
"


In [ ]:
// disabling broadcast join to reinforce the power of bucketing join

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

matchesBucketed.createOrReplaceTempView("matches")
matchDetailsBucketed.createOrReplaceTempView("match_details")

In [ ]:
// comparison of physical plans with and without bucket join

spark.sql("""
    SELECT * FROM bootcamp.match_details_bucketed mdb JOIN bootcamp.matches_bucketed md 
    ON mdb.match_id = md.match_id
    AND md.completion_date = DATE('2016-01-01')
""").explain()
// bucket join will force batch scan, avoiding shuffle, and it will have less steps to go to sortmergejoin

spark.sql("""
    SELECT * FROM match_details mdb JOIN matches md ON mdb.match_id = md.match_id AND md.completion_date = DATE('2016-01-01')
""").explain()

In [ ]:

// spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1000000000000")

// val broadcastFromThreshold = matches.as("m").join(matchDetails.as("md"), $"m.match_id" === $"md.match_id")
//   .select($"m.completion_date", $"md.player_gamertag",  $"md.player_total_kills")
//   .take(5)

// val explicitBroadcast = matches.as("m").join(broadcast(matchDetails).as("md"), $"m.match_id" === $"md.match_id")
//   .select($"md.*", split($"completion_date", " ").getItem(0).as("ds"))

// val bucketedValues = matchDetailsBucketed.as("mdb").join(matchesBucketed.as("mb"), $"mb.match_id" === $"mdb.match_id").explain()
// // .take(5)

// val values = matchDetailsBucketed.as("m").join(matchesBucketed.as("md"), $"m.match_id" === $"md.match_id").explain()

// explicitBroadcast.write.mode("overwrite").insertInto("match_details_bucketed")

// matches.withColumn("ds", split($"completion_date", " ").getItem(0)).write.mode("overwrite").insertInto("matches_bucketed")

// spark.sql(bucketedSQL)

